# Cookie Cats — статистический анализ удержания

Сравниваем удержание игроков, назначенных в `gate_40` и `gate_30`, в порядке **`gate_40` − `gate_30`**. D7 — основная метрика, D1 — дополнительная. См. [дизайн эксперимента](../docs/experiment_design.md) и [аудит данных](01_data_audit_eda.ipynb).

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from statsmodels.stats.proportion import confint_proportions_2indep, proportions_ztest

alpha = 0.05

In [2]:
data_path = Path('data/raw/cookie_cats.csv')
if not data_path.is_file():
    data_path = Path('..') / data_path

df = pd.read_csv(data_path)
df.head()

,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True


## Удержание по группам

In [3]:
retention_by_version = df.groupby('version').agg(
    players=('userid', 'size'),
    d7=('retention_7', 'mean'),
    d1=('retention_1', 'mean'),
    returned_7=('retention_7', 'sum'),
    returned_1=('retention_1', 'sum')
)

test_groups = retention_by_version.loc[['gate_40', 'gate_30']]
retention_by_version.assign(
    d1=retention_by_version['d1'].map('{:.2%}'.format),
    d7=retention_by_version['d7'].map('{:.2%}'.format),
)

,players,d7,d1,returned_7,returned_1
version,,,,,
gate_30,44700,19.02%,44.82%,8502,20034
gate_40,45489,18.20%,44.23%,8279,20119


## Основная метрика — D7

Проверяем $H_0: p_{40} - p_{30} = 0$ против $H_1: p_{40} - p_{30} \ne 0$ двусторонним z-тестом двух долей при $\alpha = 0{,}05$. Границы 95%-го доверительного интервала для разности долей считаем методом Ньюкомба.

In [4]:
d7_control = retention_by_version.loc['gate_30', 'd7']
d7_treatment = retention_by_version.loc['gate_40', 'd7']
d7_diff = d7_treatment - d7_control
d7_diff_pp = d7_diff * 100
d7_relative_pct = d7_diff / d7_control * 100

print(f'Разность D7: {d7_diff_pp:+.2f} п.п.')
print(f'Относительное изменение D7: {d7_relative_pct:+.2f}%')

Разность D7: -0.82 п.п.
Относительное изменение D7: -4.31%


In [5]:
d7_z_stat, d7_p_value = proportions_ztest(
    count=test_groups['returned_7'],
    nobs=test_groups['players'],
    value=0,
    alternative='two-sided'
)

d7_ci_low, d7_ci_high = confint_proportions_2indep(
    count1=retention_by_version.loc['gate_40', 'returned_7'],
    nobs1=retention_by_version.loc['gate_40', 'players'],
    count2=retention_by_version.loc['gate_30', 'returned_7'],
    nobs2=retention_by_version.loc['gate_30', 'players'],
    method='newcomb',
    compare='diff',
    alpha=alpha
)

print(f'Двусторонний z-тест: z = {d7_z_stat:.4f}, p-value = {d7_p_value:.5f}')
print(f'95%-й ДИ Ньюкомба для разности D7: '
      f'[{d7_ci_low * 100:+.2f}; {d7_ci_high * 100:+.2f}] п.п.')

Двусторонний z-тест: z = -3.1644, p-value = 0.00155
95%-й ДИ Ньюкомба для разности D7: [-1.33; -0.31] п.п.


В `gate_40` наблюдаемое D7 ниже на **0,82 п.п.** (относительно контроля — **−4,31%**). Это примерно **на 8 возвратов меньше на 1 000 игроков**. Отклоняем гипотезу о равенстве D7 при $p \approx 0{,}00155$. Доверительный интервал соответствует снижению на **0,31–1,33 п.п.**

## Дополнительная метрика — D1

In [6]:
d1_control = retention_by_version.loc['gate_30', 'd1']
d1_treatment = retention_by_version.loc['gate_40', 'd1']
d1_diff = d1_treatment - d1_control
d1_diff_pp = d1_diff * 100
d1_relative_pct = d1_diff / d1_control * 100

print(f'Разность D1: {d1_diff_pp:+.2f} п.п.')
print(f'Относительное изменение D1: {d1_relative_pct:+.2f}%')

Разность D1: -0.59 п.п.
Относительное изменение D1: -1.32%


In [7]:
d1_z_stat, d1_p_value = proportions_ztest(
    count=test_groups['returned_1'],
    nobs=test_groups['players'],
    value=0,
    alternative='two-sided'
)

d1_ci_low, d1_ci_high = confint_proportions_2indep(
    count1=retention_by_version.loc['gate_40', 'returned_1'],
    nobs1=retention_by_version.loc['gate_40', 'players'],
    count2=retention_by_version.loc['gate_30', 'returned_1'],
    nobs2=retention_by_version.loc['gate_30', 'players'],
    method='newcomb',
    compare='diff',
    alpha=alpha
)

print(f'Двусторонний z-тест: z = {d1_z_stat:.4f}, p-value = {d1_p_value:.5f}')
print(f'95%-й ДИ Ньюкомба для разности D1: '
      f'[{d1_ci_low * 100:+.2f}; {d1_ci_high * 100:+.2f}] п.п.')

Двусторонний z-тест: z = -1.7841, p-value = 0.07441
95%-й ДИ Ньюкомба для разности D1: [-1.24; +0.06] п.п.


Наблюдаемое D1 в `gate_40` ниже на **0,59 п.п.** 95%-й ДИ **[−1,24; +0,06] п.п.** включает ноль. При $p \approx 0{,}07441$ гипотезу о равенстве D1 на уровне 5% не отклоняем. Результат по D1 трактуем как дополнительный, его p-value приведено без поправки на множественные проверки.

## Проверка устойчивости — bootstrap для D7

В каждом из 10 000 повторов пересэмплируем игроков с возвращением отдельно внутри каждой группы, сохраняя её размер. Для бинарной метрики число возвратов в такой выборке можно сгенерировать биномиально с наблюдаемой долей возвратов.

In [8]:
seed = 42
rng = np.random.default_rng(seed=seed)
n_bootstrap = 10_000

n_gate_30 = retention_by_version.loc['gate_30', 'players']
n_gate_40 = retention_by_version.loc['gate_40', 'players']

p_gate_30 = retention_by_version.loc['gate_30', 'd7']
p_gate_40 = retention_by_version.loc['gate_40', 'd7']

gate_30_bootstrap_counts = rng.binomial(n=n_gate_30, p=p_gate_30, size=n_bootstrap)
gate_40_bootstrap_counts = rng.binomial(n=n_gate_40, p=p_gate_40, size=n_bootstrap)

bootstrap_d7_diffs = gate_40_bootstrap_counts / n_gate_40 - gate_30_bootstrap_counts / n_gate_30
bootstrap_ci_low, bootstrap_ci_high = np.quantile(bootstrap_d7_diffs, [alpha / 2, 1 - alpha / 2])

print(f'95%-й процентильный bootstrap-интервал D7: '
      f'[{bootstrap_ci_low * 100:+.2f}; {bootstrap_ci_high * 100:+.2f}] п.п.')

95%-й процентильный bootstrap-интервал D7: [-1.34; -0.31] п.п.


Процентильный bootstrap-интервал **[−1,34; −0,31] п.п.** близок к интервалу Ньюкомба **[−1,33; −0,31] п.п.** Оба метода дают интервал ниже нуля.

## Итог и ограничения

По имеющимся данным **перенос ворот на 40-й уровень не рекомендуем**. Наблюдаемое D7 в этой группе ниже, а практическая важность разницы в 0,82 п.п. требует отдельного продуктового порога. Учебный MDE использовался для расчёта выборки.

При допущении о назначении 50/50 проверка SRM дала $p = 0{,}0086$. Исходная пропорция назначения и причина перекоса неизвестны. Это ограничивает причинную интерпретацию разницы даже при согласии двух способов расчёта интервала.